# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NJ555/flyrank-ml-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from pathlib import Path
import pandas as pd

Path("data/raw").mkdir(parents=True, exist_ok=True)

csv_path = Path("data/raw/content_refresh_anonymized.csv")

if not csv_path.exists():
    !wget -O data/raw/content_refresh_anonymized.csv https://raw.githubusercontent.com/NJ555/flyrank-ml-starter/main/data/raw/content_refresh_anonymized.csv

print(csv_path.exists())

True


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
I used a classification model for this lane because the output is a 3-class action label: review_first, review_next, and monitor.

This is a tabular problem, so tree-based models fit well. I started with Logistic Regression as a simple baseline, then tried Random Forest and Gradient Boosting to see whether a stronger model improves the validation score.

I also kept the model easy to explain, because the final notebook should show not only score, but also what the model is learning from the data.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
I used a grouped split by client_id. This is a fairer test because rows from the same client can look very similar, and I do not want the same client in both train and validation.

I kept the validation set untouched during training. This makes the result closer to a real unseen-client check and avoids leakage.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Rebuild the baseline file from the raw dataset
raw_path = Path("data/raw/content_refresh_anonymized.csv")
out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_path)

def score_row(row):
    score = 0
    impressions = row.get("impressions_90d", 0)
    ctr = row.get("ctr", 0)
    pos = row.get("avg_position", 0)
    stale = row.get("days_since_last_update", 0)

    if impressions >= 3000:
        score += 3
    elif impressions >= 300:
        score += 2
    elif impressions > 0:
        score += 1

    if ctr < 1:
        score += 4
    elif ctr < 2:
        score += 2

    if pos == 0:
        score += 1
    elif pos > 50:
        score += 4
    elif pos > 20:
        score += 3
    elif pos > 10:
        score += 2

    if stale >= 180:
        score += 3
    elif stale >= 90:
        score += 1

    return score

def reason_code(row):
    ctr = row.get("ctr", 0)
    pos = row.get("avg_position", 0)
    stale = row.get("days_since_last_update", 0)

    if ctr < 1 and pos > 20 and stale >= 180:
        return "low_ctr_poor_position_stale"
    if ctr < 1 and pos > 20:
        return "low_ctr_poor_position"
    if ctr < 2 and stale >= 180:
        return "low_ctr_stale"
    if pos > 20:
        return "poor_position"
    if stale >= 180:
        return "stale_content"
    return "general_review"

def action_label(score):
    if score >= 8:
        return "review_first"
    elif score >= 5:
        return "review_next"
    return "monitor"

df["score"] = df.apply(score_row, axis=1)
df["reason_code"] = df.apply(reason_code, axis=1)
df["action_label"] = df["score"].apply(action_label)

df = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
df["rank"] = np.arange(1, len(df) + 1)

df.to_csv(out_path, index=False)
print("Saved:", out_path)

# Now load the file we just created
df = pd.read_csv(out_path)

target = "action_label"
group_col = "client_id"

drop_cols = [target, group_col]
for c in ["score", "rank", "reason_code", "content_id"]:
    if c in df.columns:
        drop_cols.append(c)

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[target]
groups = df[group_col]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

# OneHotEncoder compatibility for different sklearn versions
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe)
        ]), categorical_cols),
    ],
    remainder="drop"
)

models = {
    "Dummy most_frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample"
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

rows = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)

    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_val, preds),
        "macro_f1": f1_score(y_val, preds, average="macro"),
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False)
results_df

Saved: work/outputs/baseline_action_score.csv


,model,accuracy,macro_f1
2,Random Forest,0.997728,0.996109
3,Gradient Boosting,0.993510,0.994302
1,Logistic Regression,0.980853,0.950762
0,Dummy most_frequent,0.434529,0.201938


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
The model is strongest on rows with clear signals and weaker on cases where the labels are close together.

The main confusion is between monitor and review_next, and a smaller number of review_first cases are also close to the boundary. The most important features are CTR, average position, clicks_90d, impressions_90d, and freshness-related columns.

This means the model is learning the same kind of pattern that the Week-4 rule used, but in a more flexible way.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
best_preds = best_model.predict(X_val)

print("Best model:", best_name)
print()
print(classification_report(y_val, best_preds))
print("Confusion matrix:")
print(confusion_matrix(y_val, best_preds))

# Feature importance for Random Forest
rf_model = fitted_models["Random Forest"]
feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
importances = rf_model.named_steps["model"].feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp.head(15)

Best model: Random Forest

              precision    recall  f1-score   support

     monitor       1.00      0.98      0.99       264
review_first       1.00      1.00      1.00      2678
 review_next       1.00      1.00      1.00      3221

    accuracy                           1.00      6163
   macro avg       1.00      0.99      1.00      6163
weighted avg       1.00      1.00      1.00      6163

Confusion matrix:
[[ 260    0    4]
 [   0 2675    3]
 [   0    7 3214]]


,feature,importance
24,num__ctr,0.228124
25,num__avg_position,0.138526
6,num__clicks_90d,0.071980
68,cat__position_tier_page_1,0.067128
5,num__impressions_90d,0.044694
69,cat__position_tier_page_3_5,0.044127
13,num__days_with_impressions,0.030452
18,num__impressions_prev_30d,0.028742
23,num__days_since_last_update,0.025857
70,cat__position_tier_striking,0.022830


My best model on the validation split was Random Forest.

It performed better than the baseline on the same split and same metric, with accuracy of 0.997728 and macro F1 of 0.996109.

The model still makes a few mistakes on rows near the boundary between monitor, review_next, and review_first.

The most important features were CTR, average position, clicks_90d, impressions_90d, and freshness-related columns.

Overall, the Random Forest model is the strongest and most reliable choice for this lane.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.